# Memakai Model Fondasi Pralatih dengan KerasHub

**Buku:** *Deep Learning dengan Python* — Bab 9, Pretraining Skala Besar dan Model Fondasi

Mengunduh dan memakai model pralatih: tokenizer subkata, *fine-tuning* BERT untuk klasifikasi sentimen, dan pembangkitan teks dengan GPT-2 dengan berbagai strategi *sampling*.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±15 menit

**Isi notebook:**
- Tokenisasi subkata (WordPiece) dan efeknya pada teks bahasa Indonesia.
- `TextClassifier.from_preset(...)`: BERT siap di-*fine-tune* dari teks mentah.
- `CausalLM.from_preset("gpt2_base_en")`: *greedy*, *top-k*, dan *top-p sampling*.

In [ ]:
!pip install -q --upgrade keras-hub

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import keras_hub
print("KerasHub", keras_hub.__version__)

> Bobot model diunduh dari Kaggle Models. Bila unduhan meminta autentikasi, buat token di kaggle.com → *Settings → API*, lalu simpan sebagai *secret* `KAGGLE_USERNAME` dan `KAGGLE_KEY` di Colab.

## 1. Tokenizer subkata

In [ ]:
def dari_preset(kelas, preset, **kwargs):
    """Muat preset KerasHub; bila Kaggle Models tidak tersedia, pakai salinan di Hugging Face."""
    try:
        return kelas.from_preset(preset, **kwargs)
    except Exception as galat:
        print("Kaggle Models tidak tersedia:", str(galat)[:120], "-> memakai hf://keras/" + preset)
        return kelas.from_preset("hf://keras/" + preset, **kwargs)

tokenizer = dari_preset(keras_hub.tokenizers.Tokenizer, "bert_tiny_en_uncased")
for kalimat in ["Deep learning is fascinating.", "Pembelajaran mendalam sangat menarik."]:
    ids = keras.ops.convert_to_numpy(tokenizer(kalimat))
    print(kalimat)
    print("  ", [tokenizer.id_to_token(int(i)) for i in ids], f"({len(ids)} token)")

Kosakata BERT bahasa Inggris memecah kata-kata Indonesia menjadi banyak potongan kecil. Inilah alasan model multibahasa (atau model khusus bahasa Indonesia seperti IndoBERT) lebih efisien untuk teks Indonesia.

## 2. *Fine-tuning* BERT untuk sentimen IMDB

In [ ]:
import tarfile, pathlib, numpy as np

def siapkan_aclimdb(folder="data"):
    """Siapkan data/aclImdb/{train,test}/{neg,pos,unsup}/*.txt.
    Sumber utama: server Stanford. Bila tidak dapat dihubungi, unduh salinan
    resmi yang sama dari Hugging Face (stanfordnlp/imdb)."""
    akar = pathlib.Path(folder) / "aclImdb"
    if (akar / "train" / "pos").exists():
        return
    try:
        jalur = keras.utils.get_file(
            "aclImdb_v1.tar.gz",
            origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz")
        with tarfile.open(jalur) as berkas_tar:
            berkas_tar.extractall(folder)
    except Exception as galat:
        print("Server Stanford tidak dapat dihubungi:", galat)
        print("Memakai salinan Hugging Face (stanfordnlp/imdb) ...")
        from datasets import load_dataset
        ds = load_dataset("stanfordnlp/imdb")
        for bagian, nama in [("train", "train"), ("test", "test"), ("unsupervised", "train")]:
            for i, contoh in enumerate(ds[bagian]):
                jenis = "unsup" if bagian == "unsupervised" else ("pos" if contoh["label"] == 1 else "neg")
                tujuan = akar / nama / jenis
                tujuan.mkdir(parents=True, exist_ok=True)
                (tujuan / f"{bagian}_{i}.txt").write_text(contoh["text"], encoding="utf-8")

siapkan_aclimdb()

def baca_folder(folder, jenis=(("neg", 0), ("pos", 1))):
    teks, label = [], []
    for nama, nilai in jenis:
        for p in sorted((pathlib.Path(folder) / nama).glob("*.txt")):
            teks.append(p.read_text(encoding="utf-8"))
            label.append(nilai)
    return teks, np.array(label)

rng = np.random.default_rng(42)
def acak(teks, label):
    urutan = rng.permutation(len(teks))
    return [teks[i] for i in urutan], label[urutan]

teks_latih, y_latih = acak(*baca_folder("data/aclImdb/train"))
teks_uji, y_uji = acak(*baca_folder("data/aclImdb/test"))
print(len(teks_latih), "ulasan latih |", len(teks_uji), "ulasan uji")

In [ ]:
import tensorflow as tf

PRESET = "bert_tiny_en_uncased"      # coba "bert_small_en_uncased" bila waktu cukup
N_LATIH = 10000

pengklasifikasi = dari_preset(keras_hub.models.TextClassifier, PRESET, num_classes=2)
pengklasifikasi.compile(optimizer=keras.optimizers.Adam(5e-5),
                        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                        metrics=["accuracy"])
pengklasifikasi.summary()

ds_latih = tf.data.Dataset.from_tensor_slices((teks_latih[:N_LATIH], y_latih[:N_LATIH])).batch(32)
ds_val = tf.data.Dataset.from_tensor_slices((teks_latih[-5000:], y_latih[-5000:])).batch(64)
ds_uji = tf.data.Dataset.from_tensor_slices((teks_uji, y_uji)).batch(64)

pengklasifikasi.fit(ds_latih, validation_data=ds_val, epochs=3)
print(f"Akurasi uji: {pengklasifikasi.evaluate(ds_uji, verbose=0)[1]:.2%}")

In [ ]:
contoh = ["An absolute masterpiece, I was moved to tears.",
          "Two hours of my life I will never get back."]
logit = keras.ops.convert_to_numpy(pengklasifikasi.predict(contoh, verbose=0))
prob = keras.ops.convert_to_numpy(keras.ops.softmax(logit))
for t, p in zip(contoh, prob):
    print(f"P(positif) = {p[1]:.2f} | {t}")

## 3. Pembangkitan teks dengan GPT-2

In [ ]:
gpt2 = dari_preset(keras_hub.models.CausalLM, "gpt2_base_en")
awalan = "The most important idea in deep learning is"

for nama, sampler in [("greedy", "greedy"),
                      ("top-k (k=10)", keras_hub.samplers.TopKSampler(k=10, seed=1)),
                      ("top-p (p=0.9)", keras_hub.samplers.TopPSampler(p=0.9, seed=1))]:
    gpt2.compile(sampler=sampler)
    print(f"--- {nama} ---")
    print(gpt2.generate(awalan, max_length=60), "\n")

*Greedy* cenderung mengulang frasa yang sama. *Top-k* dan *top-p* menghasilkan teks yang lebih bervariasi. Ini analog dengan parameter suhu di notebook generasi teks LSTM (Bab 14).